In [10]:
import pandas as pd
import numpy as np

from sklearn.model_selection import LeaveOneOut
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel
from sklearn.metrics import mean_squared_error

kernel = ConstantKernel(1.0, (1e-4, 1e4)) * \
         RBF(length_scale=1.0, length_scale_bounds=(1e-8, 1e4)) + \
         WhiteKernel(noise_level=1e-3, noise_level_bounds=(1e-6, 1e1))

reg = GaussianProcessRegressor(kernel=kernel)


df = pd.read_csv("components.csv")

X = df.drop(columns=["measurement","no","id"]).values
y = df["measurement"].values


model = Pipeline([
    ("scaler", StandardScaler()),
    ("reg", reg)
])


print(X.shape)
print(X[:5])  # Print first 5 rows of X
print(y.shape)
print(y[:5]) # Print first 5 values of y

(18, 14)
[[1.00000000e+02 2.30000000e+01 2.30000000e+01 1.88800000e+01
  5.70000000e+01 3.00000000e+01 2.14000000e+01 3.26000000e+01
  2.90000000e-01 5.75000000e+02 6.03100000e-02 2.47600000e-01
  5.98500000e+00 1.44670746e+00]
 [2.00000000e+02 2.30000000e+01 2.30000000e+01 7.80000000e+01
  5.70000000e+01 3.00000000e+01 2.14000000e+01 3.26000000e+01
  2.00000000e-01 1.15000000e+03 3.65000000e-02 1.40000000e-01
  5.98500000e+00 1.44670746e+00]
 [2.00000000e+02 2.30000000e+01 5.00000000e+01 3.23000000e+01
  6.60000000e+01 3.60000000e+01 2.47000000e+01 3.90000000e+01
  2.00000000e-01 8.40000000e+02 5.15300000e-02 2.50800000e-01
  7.38000000e+00 1.42544002e+00]
 [2.00000000e+02 1.00000000e+02 5.00000000e+01 3.24000000e+01
  6.60000000e+01 3.60000000e+01 2.47000000e+01 3.90000000e+01
  2.90000000e-01 8.40000000e+02 5.69100000e-02 2.48140000e-01
  7.38000000e+00 1.42544002e+00]
 [1.00000000e+02 2.30000000e+01 5.00000000e+01 8.32000000e+00
  6.60000000e+01 3.60000000e+01 2.47000000e+01 3.9000

In [11]:
# --- LOOCV Evaluation ---
loo = LeaveOneOut()
preds = []
truth = []
stds = []

for train_idx, test_idx in loo.split(X):
    model.fit(X[train_idx], y[train_idx])
    mean, std = model.predict(X[test_idx], return_std=True)
    preds.append(mean[0])
    stds.append(std[0])
    truth.append(y[test_idx][0])

rmse = np.sqrt(mean_squared_error(truth, preds))
print("LOOCV RMSE:", rmse)

# Optional: print average uncertainty
print("Mean predictive std:", np.mean(stds))


LOOCV RMSE: 5.672680778525248
Mean predictive std: 3.5483056194754496


In [8]:
print(gpr.named_steps["gpr"].kernel_)


NameError: name 'gpr' is not defined

In [5]:
df2 = pd.read_csv("predict.csv")

X2 = df2.drop(columns=["no","id"]).values

preds = model.predict(X2)    

print("Predicted inrush currents for new data:")
for i, pred in enumerate(preds):
    print(f"Sample {i+1}: {pred}")



Predicted inrush currents for new data:
Sample 1: 24.66469800690813
Sample 2: 37.82857227849709
Sample 3: 59.6686150629687
Sample 4: 83.58726306547464
Sample 5: 103.93815245800194
Sample 6: 124.96294643790974
Sample 7: 125.0987393824563
Sample 8: 145.89960332361616
Sample 9: 122.11535300901735
Sample 10: 14.4274643651352
Sample 11: 30.049894507997
Sample 12: 40.73587238758707
Sample 13: 56.32529575938071
Sample 14: 64.2766385777263
Sample 15: 76.72059429051649
Sample 16: 70.3625285567664
Sample 17: 98.92583880715875
Sample 18: 94.31933331605796
Sample 19: 16.35250888759407
Sample 20: 32.083425251193376
Sample 21: 42.76372029768857
Sample 22: 54.35328017430535
Sample 23: 69.86231729118437
Sample 24: 82.3414633146548
Sample 25: 74.19960222277516
Sample 26: 107.52964805475786
Sample 27: 99.59743054300247
